# ECG Factorial Loss Analysis

This notebook analyzes the factorial loss-mask experiment. It supports SQLite and CSV input, decodes loss masks, compares signal and biomarker performance, tests main effects and derivative × VCG interactions, fits GEE models, and calculates clustered bootstrap confidence intervals.

> Important: if patient-level rows are unavailable, clustering by `model_id` quantifies uncertainty across models/configurations rather than patients.

In [10]:
import sqlite3
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy import stats
from IPython.display import display

warnings.filterwarnings('ignore')
RANDOM_SEED = 42
N_BOOTSTRAP = 2000
np.random.seed(RANDOM_SEED)
DB_PATH = Path('results/clinical_biomarkers_multids/clinical_metrics.db')
CSV_PATH = Path('clinical_metrics_summary.csv')

In [11]:
def load_metrics_data(db_path=DB_PATH, csv_path=CSV_PATH):
    if db_path.exists():
        try:
            with sqlite3.connect(db_path) as conn:
                tables = pd.read_sql_query(
                    "SELECT name FROM sqlite_master WHERE type='table'",
                    conn
                )
                print('SQLite tables:', tables['name'].tolist())
                if 'clinical_metrics' in tables['name'].tolist():
                    data = pd.read_sql_query(
                        'SELECT * FROM clinical_metrics', conn
                    )
                    if len(data) > 0:
                        print(f'Loaded {len(data):,} rows from SQLite')
                        return data
        except Exception as error:
            print('SQLite loading failed:', error)
    if csv_path.exists():
        data = pd.read_csv(csv_path)
        print(f'Loaded {len(data):,} rows from CSV')
        return data
    raise FileNotFoundError('No SQLite database or CSV file found.')

df = load_metrics_data()
df.columns = df.columns.astype(str).str.strip()
for column in ['dataset', 'model_id', 'architecture', 'target']:
    if column in df.columns:
        df[column] = df[column].astype(str).str.strip()

metric_columns = [
    'mae', 'pearson_r', 'r2', 'bland_bias', 'loa_low', 'loa_high',
    'auroc', 'auroc_ci_low', 'auroc_ci_high', 'auprc',
    'auprc_ci_low', 'auprc_ci_high', 'f1', 'sens', 'spec', 'ppv',
    'npv', 'adj_or', 'adj_or_ci_low', 'adj_or_ci_high',
    'pval_logistic', 'fisher_pval'
]
for column in metric_columns:
    if column in df.columns:
        df[column] = pd.to_numeric(df[column], errors='coerce')

print('Dimensions:', df.shape)
print('Models:', df['model_id'].nunique())
print('Targets:', df['target'].nunique())
display(df.head())

SQLite tables: ['clinical_metrics']
Loaded 297 rows from SQLite
Dimensions: (297, 26)
Models: 6
Targets: 58


,dataset,model_id,target,mae,pearson_r,r2,bland_bias,loa_low,loa_high,auroc,...,sens,spec,ppv,npv,adj_or,adj_or_ci_low,adj_or_ci_high,pval_logistic,fisher_pval,created_at
0,ptb_xl,f_1000000_s42,ECGFounder_Macro_150,NaN,NaN,NaN,NaN,NaN,NaN,0.849176,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-08-08 21:59:09
1,ptb_xl,f_1000000_s42,ECGFounder_NORMAL_ECG,NaN,NaN,NaN,NaN,NaN,NaN,0.817096,...,0.686397,0.754656,0.685685,0.755267,NaN,NaN,NaN,NaN,2.530665e-97,2026-08-08 21:59:09
2,ptb_xl,f_1000000_s42,ECGFounder_SINUS_RHYTHM,NaN,NaN,NaN,NaN,NaN,NaN,0.806719,...,0.879331,0.347328,0.811466,0.473958,NaN,NaN,NaN,NaN,2.593019e-29,2026-08-08 21:59:09
3,ptb_xl,f_1000000_s42,ECGFounder_SINUS_BRADYCARDIA,NaN,NaN,NaN,NaN,NaN,NaN,0.939960,...,0.968750,0.846298,0.158974,0.998894,NaN,NaN,NaN,NaN,5.668910e-46,2026-08-08 21:59:09
4,ptb_xl,f_1000000_s42,ECGFounder_ATRIAL_FIBRILLATION,NaN,NaN,NaN,NaN,NaN,NaN,0.977562,...,0.947368,0.974096,0.730964,0.996002,NaN,NaN,NaN,NaN,7.500384e-169,2026-08-08 21:59:09


In [12]:
def decode_mask(model_id):
    parts = str(model_id).split('_')
    mask = parts[1] if len(parts) > 1 else np.nan
    seed = np.nan
    if len(parts) > 2 and parts[2].replace('s', '').isdigit():
        seed = int(parts[2].replace('s', ''))
    if not isinstance(mask, str) or len(mask) != 7 or not mask.isdigit():
        return pd.Series({
            'mask': mask, 'mse_on': np.nan, 'corr_on': np.nan,
            'deriv_on': np.nan, 'vcg_on': np.nan, 'energy_on': np.nan,
            'lead_on': np.nan, 'mmd_kernel': np.nan, 'seed': seed
        })
    return pd.Series({
        'mask': mask, 'mse_on': int(mask[0]), 'corr_on': int(mask[1]),
        'deriv_on': int(mask[2]), 'vcg_on': int(mask[3]),
        'energy_on': int(mask[4]), 'lead_on': int(mask[5]),
        'mmd_kernel': int(mask[6]), 'seed': seed
    })

df = pd.concat([df, df['model_id'].apply(decode_mask)], axis=1)
mmd_names = {0: 'none', 1: 'global_rbf', 2: 'anatomical_laplacian',
             3: 'anatomical_imq', 4: 'temporal_kmeans_imq'}
df['mmd_name'] = df['mmd_kernel'].map(mmd_names)

def describe_mask(row):
    labels = []
    for column, label in [('mse_on', 'MSE'), ('corr_on', 'Correlation'),
                          ('deriv_on', 'Derivative'), ('vcg_on', 'VCG'),
                          ('energy_on', 'Energy distance'),
                          ('lead_on', 'Lead consistency')]:
        if row.get(column) == 1:
            labels.append(label)
    if row.get('mmd_kernel', 0) > 0:
        labels.append('MMD: ' + str(row.get('mmd_name')))
    return ' + '.join(labels) if labels else 'No active loss'

df['mask_description'] = df.apply(describe_mask, axis=1)
display(df[['model_id', 'mask', 'mask_description', 'mmd_name']].drop_duplicates().head(20))

signal_df = df[df['target'].str.contains('Signal', case=False, na=False)].copy()
qrs_df = df[df['target'].str.contains('QRS', case=False, na=False)].copy()
biomarker_df = df[~df['target'].str.contains('Signal|QRS', case=False, na=False)].copy()
print('Signal rows:', len(signal_df))
print('QRS rows:', len(qrs_df))
print('Biomarker rows:', len(biomarker_df))
print('Biomarker targets:', biomarker_df['target'].unique())

,model_id,mask,mask_description,mmd_name
0,f_1000000_s42,1000000,MSE,none
32,f_1000001_s42,1000001,MSE + MMD: global_rbf,global_rbf
90,f_1000002_s42,1000002,MSE + MMD: anatomical_laplacian,anatomical_laplacian
148,f_1000003_s42,1000003,MSE + MMD: anatomical_imq,anatomical_imq
206,f_1000004_s42,1000004,MSE + MMD: temporal_kmeans_imq,temporal_kmeans_imq
264,f_1000010_s42,1000010,MSE + Lead consistency,none


Signal rows: 48
QRS rows: 17
Biomarker rows: 232
Biomarker targets: ['ECGFounder_Macro_150' 'ECGFounder_NORMAL_ECG' 'ECGFounder_SINUS_RHYTHM'
 'ECGFounder_SINUS_BRADYCARDIA' 'ECGFounder_ATRIAL_FIBRILLATION'
 'ECGFounder_SINUS_TACHYCARDIA'
 'ECGFounder_PREMATURE_VENTRICULAR_COMPLEXES'
 'ECGFounder_RIGHT_BUNDLE_BRANCH_BLOCK' 'ECGFounder_SEPTAL_INFARCT'
 'ECGFounder_LEFT_ATRIAL_ENLARGEMENT' 'ECGFounder_ANTERIOR_INFARCT'
 'ECGFounder_LEFT_BUNDLE_BRANCH_BLOCK' 'ECGFounder_LATERAL_INFARCT'
 'ECGFounder_LEFT_VENTRICULAR_HYPERTROPHY' 'ECGFounder_QT_HAS_LENGTHENED'
 'ECGFounder_ATRIAL_FLUTTER' 'ECGFounder_LEFT_ANTERIOR_FASCICULAR_BLOCK'
 'ECGFounder_ANTEROSEPTAL_INFARCT'
 'ECGFounder_ELECTRONIC_ATRIAL_PACEMAKER'
 'ECGFounder_ANTEROLATERAL_INFARCT' 'ECGFounder_RIGHT_ATRIAL_ENLARGEMENT'
 'ECGFounder_INFERIOR_INFARCT'
 'ECGFounder_LEFT_POSTERIOR_FASCICULAR_BLOCK'
 'ECGFounder_WITH_1ST_DEGREE_AV_BLOCK'
 'ECGFounder_RIGHT_VENTRICULAR_HYPERTROPHY'
 'ECGFounder_SUPRAVENTRICULAR_TACHYCARDIA'
 'ECGFound

In [13]:
def summarize_by_mask(data):
    metrics = [c for c in ['mae', 'pearson_r', 'r2', 'auroc', 'auprc', 'f1']
               if c in data.columns]
    grouping = ['model_id', 'mask', 'mask_description', 'mmd_name']
    summary = data.groupby(grouping)[metrics].mean().reset_index()
    counts = data.groupby(grouping)['target'].nunique().reset_index(name='n_targets')
    return summary.merge(counts, on=grouping, how='left')

signal_summary = summarize_by_mask(signal_df)
biomarker_summary = summarize_by_mask(biomarker_df)

def rank_summary(summary, title):
    print('\n' + title)
    for metric, ascending in [('mae', True), ('pearson_r', False),
                              ('r2', False), ('auroc', False), ('auprc', False)]:
        if metric not in summary.columns:
            continue
        print('\nBest by', metric)
        display(summary.sort_values(metric, ascending=ascending).head(10).round(4))

rank_summary(signal_summary, 'Signal reconstruction rankings')
rank_summary(biomarker_summary, 'Biomarker rankings')


Signal reconstruction rankings

Best by mae


,model_id,mask,mask_description,mmd_name,mae,pearson_r,r2,auroc,auprc,f1,n_targets
3,f_1000004_s42,1000004,MSE + MMD: temporal_kmeans_imq,temporal_kmeans_imq,0.0796,0.8038,0.4632,NaN,NaN,NaN,12
2,f_1000003_s42,1000003,MSE + MMD: anatomical_imq,anatomical_imq,0.0800,0.8090,0.5206,NaN,NaN,NaN,12
1,f_1000002_s42,1000002,MSE + MMD: anatomical_laplacian,anatomical_laplacian,0.0822,0.7668,0.3742,NaN,NaN,NaN,12
0,f_1000001_s42,1000001,MSE + MMD: global_rbf,global_rbf,0.0854,0.7093,0.0421,NaN,NaN,NaN,12



Best by pearson_r


,model_id,mask,mask_description,mmd_name,mae,pearson_r,r2,auroc,auprc,f1,n_targets
2,f_1000003_s42,1000003,MSE + MMD: anatomical_imq,anatomical_imq,0.0800,0.8090,0.5206,NaN,NaN,NaN,12
3,f_1000004_s42,1000004,MSE + MMD: temporal_kmeans_imq,temporal_kmeans_imq,0.0796,0.8038,0.4632,NaN,NaN,NaN,12
1,f_1000002_s42,1000002,MSE + MMD: anatomical_laplacian,anatomical_laplacian,0.0822,0.7668,0.3742,NaN,NaN,NaN,12
0,f_1000001_s42,1000001,MSE + MMD: global_rbf,global_rbf,0.0854,0.7093,0.0421,NaN,NaN,NaN,12



Best by r2


...

,model_id,mask,mask_description,mmd_name,mae,pearson_r,r2,auroc,auprc,f1,n_targets
3,f_1000003_s42,1000003,MSE + MMD: anatomical_imq,anatomical_imq,0.0772,0.6992,0.5380,0.8625,0.4017,0.3038,43
4,f_1000004_s42,1000004,MSE + MMD: temporal_kmeans_imq,temporal_kmeans_imq,0.0727,0.6958,0.5334,0.8604,0.3966,0.3083,43
2,f_1000002_s42,1000002,MSE + MMD: anatomical_laplacian,anatomical_laplacian,0.0741,0.6333,0.4775,0.8496,0.3936,0.2914,43
1,f_1000001_s42,1000001,MSE + MMD: global_rbf,global_rbf,0.0728,0.5936,0.4240,0.8313,0.3699,0.2786,43
0,f_1000000_s42,1000000,MSE,none,NaN,NaN,NaN,0.8663,0.4126,0.3129,30
5,f_1000010_s42,1000010,MSE + Lead consistency,none,NaN,NaN,NaN,0.8398,0.3840,0.2722,30



Best by auroc


,model_id,mask,mask_description,mmd_name,mae,pearson_r,r2,auroc,auprc,f1,n_targets
0,f_1000000_s42,1000000,MSE,none,NaN,NaN,NaN,0.8663,0.4126,0.3129,30
3,f_1000003_s42,1000003,MSE + MMD: anatomical_imq,anatomical_imq,0.0772,0.6992,0.5380,0.8625,0.4017,0.3038,43
4,f_1000004_s42,1000004,MSE + MMD: temporal_kmeans_imq,temporal_kmeans_imq,0.0727,0.6958,0.5334,0.8604,0.3966,0.3083,43
2,f_1000002_s42,1000002,MSE + MMD: anatomical_laplacian,anatomical_laplacian,0.0741,0.6333,0.4775,0.8496,0.3936,0.2914,43
5,f_1000010_s42,1000010,MSE + Lead consistency,none,NaN,NaN,NaN,0.8398,0.3840,0.2722,30
1,f_1000001_s42,1000001,MSE + MMD: global_rbf,global_rbf,0.0728,0.5936,0.4240,0.8313,0.3699,0.2786,43



Best by auprc


,model_id,mask,mask_description,mmd_name,mae,pearson_r,r2,auroc,auprc,f1,n_targets
0,f_1000000_s42,1000000,MSE,none,NaN,NaN,NaN,0.8663,0.4126,0.3129,30
3,f_1000003_s42,1000003,MSE + MMD: anatomical_imq,anatomical_imq,0.0772,0.6992,0.5380,0.8625,0.4017,0.3038,43
4,f_1000004_s42,1000004,MSE + MMD: temporal_kmeans_imq,temporal_kmeans_imq,0.0727,0.6958,0.5334,0.8604,0.3966,0.3083,43
2,f_1000002_s42,1000002,MSE + MMD: anatomical_laplacian,anatomical_laplacian,0.0741,0.6333,0.4775,0.8496,0.3936,0.2914,43
5,f_1000010_s42,1000010,MSE + Lead consistency,none,NaN,NaN,NaN,0.8398,0.3840,0.2722,30
1,f_1000001_s42,1000001,MSE + MMD: global_rbf,global_rbf,0.0728,0.5936,0.4240,0.8313,0.3699,0.2786,43


In [14]:
# Main-effects and derivative × VCG interaction regression

def fit_interaction_model(data, outcome, title):
    required = [outcome, 'mse_on', 'corr_on', 'deriv_on', 'vcg_on']
    missing = [c for c in required if c not in data.columns]
    if missing:
        print(title, 'missing:', missing)
        return None
    analysis = data.dropna(subset=required).copy()
    predictors = ['mse_on', 'corr_on', 'deriv_on', 'vcg_on']
    predictors = [p for p in predictors if analysis[p].nunique() > 1]
    if len(analysis) < 20 or not predictors:
        print(title, 'has insufficient data or no varying predictors.')
        return None
    terms = predictors.copy()
    if 'deriv_on' in predictors and 'vcg_on' in predictors:
        terms.append('deriv_on:vcg_on')
    formula = outcome + ' ~ ' + ' + '.join(terms)
    if analysis['target'].nunique() > 1:
        formula += ' + C(target)'
    model = smf.ols(formula, data=analysis).fit(cov_type='HC3')
    print('\n' + title)
    print('Formula:', formula)
    display(model.summary().tables[1])
    return model

signal_mae_model = fit_interaction_model(signal_df, 'mae', 'Signal MAE model')
signal_pearson_model = fit_interaction_model(signal_df, 'pearson_r', 'Signal Pearson model')
biomarker_auroc_model = fit_interaction_model(biomarker_df, 'auroc', 'Biomarker AUROC model')

Signal MAE model has insufficient data or no varying predictors.
Signal Pearson model has insufficient data or no varying predictors.
Biomarker AUROC model has insufficient data or no varying predictors.


In [15]:
# GEE repeated-measures model

def fit_gee(data, outcome='mae', cluster='model_id'):
    required = [outcome, cluster, 'mse_on', 'corr_on', 'deriv_on', 'vcg_on']
    missing = [c for c in required if c not in data.columns]
    if missing:
        print('GEE missing:', missing)
        return None
    analysis = data.dropna(subset=required).copy()
    predictors = ['mse_on', 'corr_on', 'deriv_on', 'vcg_on']
    predictors = [p for p in predictors if analysis[p].nunique() > 1]
    if not predictors:
        print('No varying predictors available for GEE.')
        return None
    formula = outcome + ' ~ ' + ' + '.join(predictors)
    if analysis['target'].nunique() > 1:
        formula += ' + C(target)'
    model = smf.gee(
        formula, groups=analysis[cluster], data=analysis,
        family=sm.families.Gaussian(),
        cov_struct=sm.cov_struct.Exchangeable()
    ).fit()
    print('Formula:', formula)
    print('Cluster:', cluster)
    display(model.summary().tables[1])
    return model

qrs_gee = fit_gee(qrs_df, outcome='mae', cluster='model_id')

No varying predictors available for GEE.


In [16]:
# Clustered percentile bootstrap and BCa bootstrap

def cluster_bootstrap(data, cluster, outcome, stat_fn=np.mean, n_boot=2000, seed=42):
    clean = data.dropna(subset=[cluster, outcome]).copy()
    clusters = clean[cluster].unique()
    if len(clusters) < 5:
        raise ValueError('At least five clusters are recommended.')
    rng = np.random.default_rng(seed)
    observed = stat_fn(clean[outcome].to_numpy())
    boot = []
    for _ in range(n_boot):
        sampled = rng.choice(clusters, size=len(clusters), replace=True)
        pieces = [clean[clean[cluster] == c] for c in sampled]
        sample = pd.concat(pieces, ignore_index=True)
        boot.append(stat_fn(sample[outcome].to_numpy()))
    return observed, np.asarray(boot)

def bca_ci(data, cluster, outcome, stat_fn=np.mean, n_boot=2000, alpha=0.05, seed=42):
    clean = data.dropna(subset=[cluster, outcome]).copy()
    clusters = clean[cluster].unique()
    observed, boot = cluster_bootstrap(clean, cluster, outcome, stat_fn, n_boot, seed)
    prop = np.clip(np.mean(boot < observed), 1/(2*n_boot), 1-1/(2*n_boot))
    z0 = stats.norm.ppf(prop)
    jack = []
    for removed in clusters:
        remaining = clean[clean[cluster] != removed]
        jack.append(stat_fn(remaining[outcome].to_numpy()))
    jack = np.asarray(jack)
    jack_mean = jack.mean()
    denominator = 6 * np.sum((jack_mean - jack) ** 2) ** 1.5
    acceleration = (np.sum((jack_mean - jack) ** 3) / denominator
                    if denominator != 0 else 0.0)
    z_low = stats.norm.ppf(alpha / 2)
    z_high = stats.norm.ppf(1 - alpha / 2)
    def adjusted(z):
        return stats.norm.cdf(z0 + (z0 + z) / (1 - acceleration * (z0 + z)))
    lower = np.quantile(boot, adjusted(z_low))
    upper = np.quantile(boot, adjusted(z_high))
    return {'estimate': observed, 'ci_lower': lower, 'ci_upper': upper,
            'z0': z0, 'acceleration': acceleration,
            'n_clusters': len(clusters), 'n_bootstrap': n_boot}

bootstrap_results = []
for group_name, group_data in {'QRS': qrs_df, 'Signal': signal_df,
                               'Biomarker': biomarker_df}.items():
    for outcome in ['mae', 'pearson_r', 'r2', 'auroc']:
        if outcome not in group_data.columns or len(group_data) == 0:
            continue
        try:
            result = bca_ci(group_data, 'model_id', outcome,
                            n_boot=N_BOOTSTRAP, seed=RANDOM_SEED)
            result.update({'analysis_group': group_name, 'outcome': outcome})
            bootstrap_results.append(result)
        except Exception as error:
            print(group_name, outcome, error)

bootstrap_results_df = pd.DataFrame(bootstrap_results)
display(bootstrap_results_df.round(4))

Signal mae At least five clusters are recommended.
Signal pearson_r At least five clusters are recommended.
Signal r2 At least five clusters are recommended.
Signal auroc At least five clusters are recommended.
Biomarker mae At least five clusters are recommended.
Biomarker pearson_r At least five clusters are recommended.
Biomarker r2 At least five clusters are recommended.


,estimate,ci_lower,ci_upper,z0,acceleration,n_clusters,n_bootstrap,analysis_group,outcome
0,0.0000,0.0000,0.0000,-3.4808,0.0000,5,2000,QRS,mae
1,1.0000,1.0000,1.0000,-3.4808,-0.0745,5,2000,QRS,pearson_r
2,1.0000,1.0000,1.0000,-3.4808,0.0000,5,2000,QRS,r2
3,0.7035,0.6484,0.7231,-0.1295,-0.0944,6,2000,QRS,auroc
4,0.8516,0.8394,0.8604,-0.0589,-0.0296,6,2000,Biomarker,auroc


In [17]:
# Save outputs

OUTPUT_DIR = Path('results/factorial_analysis')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

signal_summary.to_csv(OUTPUT_DIR / 'signal_mask_summary.csv', index=False)
biomarker_summary.to_csv(OUTPUT_DIR / 'biomarker_mask_summary.csv', index=False)
bootstrap_results_df.to_csv(OUTPUT_DIR / 'bca_bootstrap_results.csv', index=False)

print('Saved outputs to:', OUTPUT_DIR.resolve())

Saved outputs to: /home/mithunmanivannan/projects/benchmarking_loss_functions_ecg_reconstruction/results/factorial_analysis
